In [1]:
# ============================================================
# Notebook    : 05_case_b_car_insurance_eda.ipynb
# Description : Case B — car_insurance (Kaggle) EDA & preprocessing
#               - Cross-sectional (single-timestamp) dataset,
#                 NOT longitudinal — no IDpol-style sequence
#                 structure exists here
#               - Splits features into two groups per design
#                 decision:
#                   B1 (demographics only)
#                   B2 (B1 + behavioral history: SPEEDING_
#                       VIOLATIONS, DUIS, PAST_ACCIDENTS)
#               - Label: OUTCOME (already binary in source data)
# ============================================================


# ============================================================
# 1. Common imports
# ============================================================
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


# ============================================================
# 2. Load (from notebook 00 output)
# ============================================================
df = pd.read_csv("data/car_insurance.csv")

print(f"[CHECK 1] Shape: {df.shape}")
print(df.head())
print(f"\n[CHECK 1] dtypes:")
print(df.dtypes)


# ============================================================
# 3. Missing value check — CREDIT_SCORE and ANNUAL_MILEAGE
#    looked suspicious in the raw sample output from notebook 00
# ============================================================
print(f"\n[CHECK 2] Missing values per column:")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_summary[missing_summary["missing_count"] > 0].to_string())

if missing_summary["missing_count"].sum() == 0:
    print("  (no missing values found)")


# ============================================================
# 4. Label check — OUTCOME
# ============================================================
print(f"\n[CHECK 3] OUTCOME distribution:")
print(df["OUTCOME"].value_counts())
print(f"Positive rate: {df['OUTCOME'].mean()*100:.2f}%")


# ============================================================
# 5. Categorical column inventory — needed to decide encoding
#    strategy before LightGBM (native categorical support)
# ============================================================
cat_candidates = df.select_dtypes(include=["object"]).columns.tolist()
print(f"\n[CHECK 4] Object-dtype columns: {cat_candidates}")
for col in cat_candidates:
    print(f"\n  {col} — {df[col].nunique()} unique values:")
    print(f"    {df[col].value_counts().to_dict()}")


# ============================================================
# 6. Feature grouping — B1 (demographics only) vs
#    B2 (B1 + behavioral/violation history)
#
#    ID and POSTAL_CODE excluded from both:
#      - ID is a pure identifier, no signal
#      - POSTAL_CODE is high-cardinality and would need separate
#        handling (target encoding, geographic clustering, etc.)
#        that's out of scope for this comparison; flagged as a
#        future extension, not silently dropped without note
# ============================================================
DEMOGRAPHIC_COLS = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION",
    "INCOME", "CREDIT_SCORE", "VEHICLE_OWNERSHIP", "VEHICLE_YEAR",
    "MARRIED", "CHILDREN", "ANNUAL_MILEAGE", "VEHICLE_TYPE",
]
BEHAVIORAL_COLS = ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]
EXCLUDED_COLS   = ["ID", "POSTAL_CODE"]
LABEL_COL       = "OUTCOME"

print(f"\n[CHECK 5] Feature grouping:")
print(f"  B1 (demographics only)      : {len(DEMOGRAPHIC_COLS)} cols — {DEMOGRAPHIC_COLS}")
print(f"  B2 additional (behavioral)  : {len(BEHAVIORAL_COLS)} cols — {BEHAVIORAL_COLS}")
print(f"  Excluded                    : {EXCLUDED_COLS} (identifier / high-cardinality geo)")

all_expected = set(DEMOGRAPHIC_COLS + BEHAVIORAL_COLS + EXCLUDED_COLS + [LABEL_COL])
all_actual   = set(df.columns)
unaccounted  = all_actual - all_expected
if unaccounted:
    print(f"\n  !! WARNING: columns present in data but not categorized: {unaccounted}")
else:
    print(f"\n  All {len(df.columns)} columns accounted for.")


# ============================================================
# 7. Handle missing values
#    - Numeric (CREDIT_SCORE, ANNUAL_MILEAGE likely candidates):
#      median imputation, with a companion "_was_missing" flag
#      so the model can distinguish "imputed" from "observed"
#      (standard practice — avoids silently treating imputed
#      values as if they were real measurements)
# ============================================================
numeric_cols_with_na = missing_summary[missing_summary["missing_count"] > 0].index.tolist()

for col in numeric_cols_with_na:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col + "_was_missing"] = df[col].isna().astype(int)
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"[CHECK 6] {col}: filled {df[col + '_was_missing'].sum()} missing "
              f"values with median={median_val:.4f}, flag column added")
    else:
        print(f"[CHECK 6] !! {col} is non-numeric and has missing values — "
              f"needs separate handling, not auto-imputed")


# ============================================================
# 8. Sanity check — confirm no missing values remain in the
#    columns we plan to use as features
# ============================================================
feature_cols_check = DEMOGRAPHIC_COLS + BEHAVIORAL_COLS
remaining_na = df[feature_cols_check].isna().sum()
remaining_na = remaining_na[remaining_na > 0]
print(f"\n[CHECK 7] Remaining NA in feature columns after imputation:")
if len(remaining_na) > 0:
    print(remaining_na)
else:
    print("  None — all clear.")


# ============================================================
# 9. Save
# ============================================================
df.to_csv("data/car_insurance_preprocessed.csv", index=False)
print(f"\nSaved: data/car_insurance_preprocessed.csv")


# ============================================================
# 10. Summary
# ============================================================
print("\n===== Case B Preprocessing Summary =====")
print(f"Total rows              : {len(df):,}")
print(f"Positive rate (OUTCOME) : {df['OUTCOME'].mean()*100:.2f}%")
print(f"B1 feature count        : {len(DEMOGRAPHIC_COLS)}")
print(f"B2 feature count        : {len(DEMOGRAPHIC_COLS) + len(BEHAVIORAL_COLS)}")
print(f"Missing value columns   : {numeric_cols_with_na if numeric_cols_with_na else 'none'}")
print(f"Output file             : data/car_insurance_preprocessed.csv")
print(f"NOTE: this dataset is CROSS-SECTIONAL — no train/val/test")
print(f"      IDpol-sequence split exists yet. Row-level random")
print(f"      split will be done in the next notebook (no leakage")
print(f"      risk since each row is an independent policyholder).")
print("===========================================")

[CHECK 1] Shape: (10000, 19)
       ID    AGE  GENDER      RACE DRIVING_EXPERIENCE    EDUCATION  \
0  569520    65+  female  majority               0-9y  high school   
1  750365  16-25    male  majority               0-9y         none   
2  199901  16-25  female  majority               0-9y  high school   
3  478866  16-25    male  majority               0-9y   university   
4  731664  26-39    male  majority             10-19y         none   

          INCOME  CREDIT_SCORE  VEHICLE_OWNERSHIP VEHICLE_YEAR  MARRIED  \
0    upper class      0.629027                1.0   after 2015      0.0   
1        poverty      0.357757                0.0  before 2015      0.0   
2  working class      0.493146                1.0  before 2015      0.0   
3  working class      0.206013                1.0  before 2015      0.0   
4  working class      0.388366                1.0  before 2015      0.0   

   CHILDREN  POSTAL_CODE  ANNUAL_MILEAGE VEHICLE_TYPE  SPEEDING_VIOLATIONS  \
0       1.0        10